# Lecture 10 Lab — Forecasting Tool Wear with an LSTM
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, matplotlib, scikit-learn, PyTorch |
| **Hardware** | CPU is fine — the whole lab trains in two to three minutes |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID


### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. The Problem

A CNC machining centre runs continuously. Its carbide cutting insert wears a little with every part it cuts, and flank wear (VB, in millimetres) follows a well-known three-phase curve:

1. **Break-in** — rapid but *decelerating*, as the fresh edge micro-fractures into shape.
2. **Steady state** — slow and roughly linear. Most of the tool's useful life.
3. **Accelerated breakdown** — rapid and *accelerating*. The edge is collapsing.

When VB crosses **0.55 mm** the operator swaps the insert and wear resets to zero. Your machine log therefore contains many tool lives back to back.

You cannot measure VB during production — that needs a microscope and a stopped machine. You *can* measure four sensor channels continuously. Your task:

> **Given the last 30 readings, forecast what VB will be 20 parts from now.**

Forecast too low and you run a worn tool into a scrap part. Forecast too high and you throw away a tool with life left in it.

### Why this week needs a new kind of model

Week 9's CNN exploited *spatial* structure: neighbouring pixels are related. A time series has *temporal* structure — and the order of the readings is itself information. Consider two windows containing exactly the same thirty sensor values:

- **high values, then low** — the tool was just replaced. Future wear is near zero.
- **low values, then high** — a fresh tool is wearing in. Future wear is substantial.

Same numbers, opposite forecasts. Any model that treats a window as an unordered bag of values is blind to the difference. That is what today is about.

### Learning objectives

- Turn a continuous sensor log into supervised `(window → future value)` training pairs without leaking the future.
- Explain how a recurrent network carries a hidden state along a sequence, and why LSTM gates fix the vanishing-gradient problem.
- Build and train an LSTM forecaster in PyTorch, getting the tensor shapes right.
- Design and run an experiment that proves your model is genuinely using temporal order.
- Judge forecast error against a maintenance-decision threshold, not just as a number.

## 2. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_absolute_error

torch.manual_seed(s_id)
np.random.seed(s_id)
torch.set_num_threads(2)
print("PyTorch", torch.__version__)

## 3. Part A — Simulating the Machine Log

`tool_life_curve` produces one insert's three-phase wear curve, with per-tool variation in how long each phase lasts. `make_machine_log` chains many tool lives together, cutting each one off at the 0.55 mm replacement limit, and generates the four sensor channels.

The sensors depend on wear **and** on per-part cutting conditions (depth of cut, feed rate) that change randomly from part to part. That confounding is deliberate: it means a single reading is a poor estimate of wear, so the model has to use the window.

In [ ]:
WINDOW   = 60      # readings the model sees
HORIZON  = 20      # how far ahead we forecast
VB_LIMIT = 0.55    # mm - insert is replaced when wear crosses this
CHANNELS = ["spindle_current_A", "vibration_rms_g", "acoustic_emission", "cutting_force_N"]


def tool_life_curve(rng):
    """Flank wear VB over one insert's life (three phases, per-tool variation)."""
    n = int(rng.integers(150, 210))
    t = np.arange(n) / n
    breakin   = rng.uniform(0.06, 0.10)   # wear accumulated during break-in
    t1        = rng.uniform(0.12, 0.20)   # break-in ends
    t2        = rng.uniform(0.70, 0.80)   # accelerated breakdown begins
    rate      = rng.uniform(0.20, 0.28)   # steady-state wear rate

    wear = np.empty(n)
    for i, ti in enumerate(t):
        if ti < t1:                                        # decelerating
            wear[i] = breakin * np.sqrt(ti / t1)
        elif ti < t2:                                      # linear
            wear[i] = breakin + rate * (ti - t1)
        else:                                              # accelerating
            wear[i] = breakin + rate * (t2 - t1) + 6.0 * (ti - t2) ** 1.9
    wear = wear + rng.normal(0, 0.0015, n)                 # measurement jitter
    return np.maximum.accumulate(np.clip(wear, 0, None))   # wear never decreases


def make_machine_log(n_tools, seed):
    """Chain many tool lives into one continuous log, swapping at VB_LIMIT."""
    rng = np.random.default_rng(seed)
    lives = []
    for _ in range(n_tools):
        w = tool_life_curve(rng)
        lives.append(w[w < VB_LIMIT])        # the swap happens at the limit
    vb = np.concatenate(lives)
    T = len(vb)

    depth = rng.uniform(0.8, 1.6, T)         # per-part cutting conditions
    feed  = rng.uniform(0.10, 0.22, T)
    sensors = np.stack([
        6.0 + 9.0 * vb + 2.2 * depth               + rng.normal(0, 0.35,  T),
        0.25 + 1.9 * vb ** 1.5 + 0.30 * depth      + rng.normal(0, 0.045, T),
        1.0 + 5.5 * vb + 11.0 * feed               + rng.normal(0, 0.28,  T),
        180.0 + 420.0 * vb + 150.0 * depth + 300.0 * feed + rng.normal(0, 14.0, T),
    ], axis=1).astype(np.float32)
    return sensors, vb.astype(np.float32)


sensors_train, vb_train = make_machine_log(40, seed=s_id)
sensors_test,  vb_test  = make_machine_log(14, seed=s_id + 1)

print("Student ID : "+str(s_id)) # don't chage this code!
print(f"training log: {len(vb_train):,} parts   test log: {len(vb_test):,} parts")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
span = slice(0, 700)
axes[0].plot(vb_train[span], color="#1B2A41", linewidth=1.5)
axes[0].axhline(VB_LIMIT, color="#FF6A39", linestyle="--", linewidth=1.2, label=f"replacement limit ({VB_LIMIT} mm)")
axes[0].set_ylabel("flank wear VB (mm)"); axes[0].legend(fontsize=8)
axes[0].set_title("Machine log: several tool lives back to back (the sawtooth is tool changes)")
axes[1].plot(sensors_train[span, 0], color="#3E5C76", linewidth=0.9, label=CHANNELS[0])
axes[1].plot(sensors_train[span, 1] * 6, color="#FF6A39", linewidth=0.9, label=CHANNELS[1] + " (x6)")
axes[1].set_xlabel("part number"); axes[1].set_ylabel("sensor reading"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

> The wear trace is a **sawtooth**: wear climbs, the insert is swapped, wear drops to zero. The sensor traces below follow it, but noisily — the scatter comes from cutting conditions changing part to part. Notice you can *see* the tool changes in the sensors, but only because you can see the readings in order.

## 4. Part B — Turning a Log into Supervised Examples

A time series is not yet a supervised dataset. We slide a window along the log: the features are `WINDOW` consecutive readings, and the target is the wear value `HORIZON` steps after the end of that window.

Fill in the `TODO`. Watch the indices — off-by-one errors here are the classic way to accidentally leak the answer into the input.

We also record, for each window, whether a tool change falls inside the window or the forecast horizon. **That flag is for analysis only** — it is never given to any model.

In [ ]:
def make_windows(sensors, vb):
    """Slide a window along the log, producing (X, y, spans_tool_change)."""
    X, y, spans = [], [], []
    for start in range(0, len(vb) - WINDOW - HORIZON, 2):   # stride 2
        end = start + WINDOW                                 # window is [start, end)

        # TODO: append to X the sensor readings for this window, and to y the
        # wear value HORIZON steps after the LAST reading in the window.
        #   - the window's last reading is at index end - 1
        #   - so the forecast target sits at index (end - 1) + HORIZON
        X.append(None)
        y.append(None)

        segment = vb[start:end + HORIZON]
        spans.append(1 if np.any(np.diff(segment) < -0.05) else 0)   # a reset happened
    return np.stack(X), np.array(y, dtype=np.float32), np.array(spans)


X_train, y_train, _          = make_windows(sensors_train, vb_train)
X_test,  y_test,  spans_test = make_windows(sensors_test,  vb_test)

assert X_train.ndim == 3 and X_train.shape[1:] == (WINDOW, len(CHANNELS)), \
    f"X should be (n_windows, {WINDOW}, {len(CHANNELS)}) but is {X_train.shape} - check the TODO above"
assert not np.isnan(y_train).any(), "y_train contains NaN - the TODO above is not filled in yet"
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"X_train {X_train.shape}   (windows, time steps, channels)")
print(f"y_train {y_train.shape}")
print(f"test windows touching a tool change: {spans_test.mean() * 100:.0f}%")

> **Why 60 readings?** The window has to be long enough to contain the evidence. A tool change plus enough of the following tool life to establish a new trend does not fit in 30 readings. Window length is a genuine design decision — §10 asks you to test it.

> **Why split by log, not by window.** The training and test windows come from *different machine logs*, generated with different seeds. If we had pooled all windows and split them randomly, neighbouring windows — which overlap by 28 of their 30 readings — would land on both sides of the split. The test score would then be measuring memorization, not forecasting. This is the Week 6 leakage rule in its time-series form, and it is the most common mistake in applied forecasting.

In [ ]:
# Normalize with TRAINING statistics only (the rule has not changed since Week 2)
mu = X_train.reshape(-1, len(CHANNELS)).mean(axis=0)
sd = X_train.reshape(-1, len(CHANNELS)).std(axis=0)
X_train_n = ((X_train - mu) / sd).astype(np.float32)
X_test_n  = ((X_test  - mu) / sd).astype(np.float32)
print("Student ID : "+str(s_id)) # don't chage this code!
print("normalized per-channel means (train):", X_train_n.reshape(-1, 4).mean(0).round(3))

## 5. Part C — The Order-Blind Baseline

Before building a sequence model, establish what a model that *cannot* see order achieves. We summarise each window by four permutation-invariant statistics per channel — mean, standard deviation, min and max — and train an MLP on those 16 numbers.

These statistics are identical no matter how the time steps are arranged. This baseline is blind to order **by construction**, which makes it the honest thing to beat.

In [ ]:
def summary_features(X):
    """Permutation-invariant window summary: identical for any ordering."""
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1)], axis=1)


def train_model(model, X, y, epochs, lr=3e-3, batch_size=64, seed=0):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1)),
                        batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
    return model


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).squeeze(1).numpy()


def mae_um(y_true, y_pred):
    """Mean absolute error in micrometres - the unit a machinist thinks in."""
    return mean_absolute_error(y_true, y_pred) * 1000

In [ ]:
S_train, S_test = summary_features(X_train_n), summary_features(X_test_n)
s_mu, s_sd = S_train.mean(0), S_train.std(0) + 1e-8
S_train, S_test = ((S_train - s_mu) / s_sd).astype(np.float32), ((S_test - s_mu) / s_sd).astype(np.float32)

SEEDS = (s_id, s_id+1, s_id+2)   # Week 6 rule: one run is a number, several runs are evidence

def make_mlp():
    return nn.Sequential(
        nn.Linear(S_train.shape[1], 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    )

mlp_models, mlp_scores = [], []
for seed in SEEDS:
    torch.manual_seed(seed)
    m = train_model(make_mlp(), S_train, y_train, epochs=40, seed=seed)
    mlp_models.append(m)
    mlp_scores.append(mae_um(y_test, predict(m, S_test)))

print("Student ID : "+str(s_id)) # don't chage this code!
print("MLP on summary statistics")
print(f"  per-seed MAE: {[round(v, 1) for v in mlp_scores]} um")
print(f"  median       : {sorted(mlp_scores)[1]:.1f} um")

## 6. Part D — The Recurrent Model

An LSTM reads the window one time step at a time, carrying a hidden state forward. After the last reading, that hidden state is a summary of the whole sequence *in order* — and we map it to a forecast.

Fill in the `TODO` in `forward`. The crux is which time step's output to use.

`nn.LSTM(..., batch_first=True)` returns `out` with shape `(batch, seq_len, hidden)` — one output vector per time step. For a many-to-one forecast we want the output after the network has seen the **entire** sequence, which is the **last** time step.

In [ ]:
class WearLSTM(nn.Module):
    def __init__(self, n_channels=4, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_channels, hidden, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)       # out: (batch, seq_len, hidden)

        # TODO: select the output at the LAST time step, giving shape (batch, hidden),
        # then pass it through self.head to produce the forecast.
        # Hint: out[:, <which index?>, :]
        last_step = None
        return self.head(last_step)


torch.manual_seed(s_id)
lstm = WearLSTM(hidden=64)
n_params = sum(p.numel() for p in lstm.parameters() if p.requires_grad)
print(lstm)
print(f"\nLSTM parameters: {n_params:,}")

# shape check before training - cheap, and catches most sequence-model bugs
demo = torch.tensor(X_train_n[:4])
print(f"input  {tuple(demo.shape)}  ->  output {tuple(lstm(demo).shape)}   (expect (4, 1))")

The cell below is training cell. It may take about 1-2 mins to finish.

In [ ]:
import time
t0 = time.time()
lstm_models, lstm_scores = [], []
for seed in SEEDS:
    torch.manual_seed(seed)
    m = train_model(WearLSTM(hidden=64), X_train_n, y_train, epochs=50, lr=1.5e-3, seed=seed)
    lstm_models.append(m)
    lstm_scores.append(mae_um(y_test, predict(m, X_test_n)))
    
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"LSTM on ordered sequences   ({time.time()-t0:.0f}s for {len(SEEDS)} seeds)")
print(f"  per-seed MAE: {[round(v, 1) for v in lstm_scores]} um")
print(f"  median       : {sorted(lstm_scores)[1]:.1f} um")

# keep the MEDIAN seed for the rest of the lab - never the best one
median_idx = int(np.argsort(lstm_scores)[1])
lstm, pred_lstm = lstm_models[median_idx], predict(lstm_models[median_idx], X_test_n)
mlp,  pred_mlp  = mlp_models[median_idx],  predict(mlp_models[median_idx],  S_test)
print(f"\n  using seed {SEEDS[median_idx]} (the median run) for all analysis below")

## 7. Part E — Does It Actually Use the Order?

A better score is not proof that the model learned temporal structure — it has more parameters and a different architecture, so it might simply be a stronger function approximator.

Here is a decisive test. Take the **already-trained** LSTM and feed it test windows whose time steps have been **randomly permuted**. The multiset of values in each window is unchanged; only the order is destroyed.

- If the model relies on order, its error should rise sharply.
- The summary-statistics MLP should be **completely unaffected**, because its features are permutation-invariant by construction. That makes it a built-in control for the experiment.

Fill in the `TODO`.

In [ ]:
rng_shuffle = np.random.default_rng(s_id)

# TODO: build X_test_shuffled by randomly permuting the TIME STEPS within each
# test window, leaving the channels and the set of values untouched.
# Hint: for each window w (shape (WINDOW, 4)), use w[rng_shuffle.permutation(WINDOW)]
X_test_shuffled = None



assert X_test_shuffled.shape == X_test_n.shape, "shuffling must not change the shape"
assert np.allclose(np.sort(X_test_shuffled[:, :, 0], axis=1), np.sort(X_test_n[:, :, 0], axis=1)), \
    "the set of values in each window must be unchanged - only their order"
print("Student ID : "+str(s_id)) # don't chage this code!
print("Shuffle verified: same values, different order.")

In [ ]:
pred_lstm_shuf = predict(lstm, X_test_shuffled)

S_shuf = summary_features(X_test_shuffled)
S_shuf = ((S_shuf - s_mu) / s_sd).astype(np.float32)
pred_mlp_shuf = predict(mlp, S_shuf)

print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'':28}{'ordered':>10}{'shuffled':>11}")
print(f"{'LSTM (sequence model)':28}{mae_um(y_test, pred_lstm):>9.1f}{mae_um(y_test, pred_lstm_shuf):>11.1f}  um")
print(f"{'MLP (summary statistics)':28}{mae_um(y_test, pred_mlp):>9.1f}{mae_um(y_test, pred_mlp_shuf):>11.1f}  um")
print(f"\nLSTM error multiplied by {mae_um(y_test, pred_lstm_shuf)/mae_um(y_test, pred_lstm):.1f}x when the order was destroyed.")
print("MLP error unchanged - its features cannot see order in the first place.")

## 8. Part F — Where the Difference Lives

The overall averages hide the interesting part. Split the test windows by whether a **tool change** falls inside the window or the forecast horizon — the situation where order carries the most information.

Remember: no model was ever told which windows these are.

In [ ]:
rows = {
    "MLP summaries (ordered)":  pred_mlp,
    "MLP summaries (shuffled)": pred_mlp_shuf,
    "LSTM (ordered)":           pred_lstm,
    "LSTM (shuffled)":          pred_lstm_shuf,
}
print(f"{'':30}{'all':>8}{'no tool change':>17}{'tool change':>14}")
results = {}
for name, p in rows.items():
    a = mae_um(y_test, p)
    b = mae_um(y_test[spans_test == 0], p[spans_test == 0])
    c = mae_um(y_test[spans_test == 1], p[spans_test == 1])
    results[name] = (a, b, c)
    print(f"{name:30}{a:>8.1f}{b:>17.1f}{c:>14.1f}")
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"\n({int((spans_test==1).sum())} of {len(spans_test)} test windows touch a tool change)")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
labels = list(results.keys())
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.bar(x - w/2, [results[k][1] for k in labels], w, color="#3E5C76", label="no tool change")
ax.bar(x + w/2, [results[k][2] for k in labels], w, color="#FF6A39", label="tool change in window/horizon")
ax.set_ylabel("MAE (µm)"); ax.set_xticks(x)
ax.set_xticklabels([l.replace(" (", "\n(") for l in labels], fontsize=9)
ax.legend(fontsize=9); ax.set_title("Forecast error by window type")
plt.tight_layout(); plt.show()

> **Read the two MLP rows first.** They are *identical* — shuffling the time steps changes its error by less than a micrometre, because its features cannot represent order in the first place. That makes it a built-in control for this experiment.
>
> **Now the LSTM rows.** Ordered, its error on tool-change windows is modest. Shuffled, that same number explodes by roughly eight times. The information it was relying on was the **order**, and tool changes are where that information is worth the most.
>
> **One number that looks odd:** the MLP's error on tool-change windows is *lower* than on ordinary ones — and lower than the LSTM's. Do not read that as the MLP winning. The two subsets are not equally hard: a window containing a tool change usually has a target close to zero, because the insert was just replaced, and predicting "near zero" is easy. Compare models *within* a column, and compare ordered against shuffled *within* a row. Comparing across columns compares two different problems.

## 9. Part G — Is the Forecast Good Enough to Act On?

A MAE in micrometres is not yet an engineering answer. The decision this model supports is: *should the operator change the insert within the next 20 parts?* Judge it that way.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

axes[0].scatter(y_test, pred_lstm, s=5, alpha=0.3, color="#FF6A39")
lims = [0, max(y_test.max(), pred_lstm.max()) * 1.05]
axes[0].plot(lims, lims, color="#1B2A41", linestyle="--", linewidth=1.2)
axes[0].axhline(0.35, color="#6B4F9C", linewidth=1, linestyle=":")
axes[0].axvline(0.35, color="#6B4F9C", linewidth=1, linestyle=":")
axes[0].set_xlabel("actual VB in 20 parts (mm)"); axes[0].set_ylabel("forecast VB (mm)")
axes[0].set_title("LSTM forecast vs. actual")

start = 300
seg = slice(start, start + 320)
axes[1].plot(y_test[seg], color="#1B2A41", linewidth=1.6, label="actual")
axes[1].plot(pred_lstm[seg], color="#FF6A39", linewidth=1.3, label="LSTM forecast")
axes[1].axhline(VB_LIMIT, color="#6B4F9C", linestyle="--", linewidth=1, label="replacement limit")
axes[1].set_xlabel("test window index"); axes[1].set_ylabel("VB (mm)")
axes[1].legend(fontsize=8); axes[1].set_title("Forecast tracking across tool changes")
plt.tight_layout(); plt.show()

In [ ]:
# Turn the forecast into the maintenance decision it is meant to support
PRE_ALARM = 0.35      # mm - "this insert is entering its last stretch, schedule a change"
actual_due   = y_test    >= PRE_ALARM
forecast_due = pred_lstm >= PRE_ALARM

tp = int((actual_due & forecast_due).sum())
fn = int((actual_due & ~forecast_due).sum())
fp = int((~actual_due & forecast_due).sum())
tn = int((~actual_due & ~forecast_due).sum())

print("Student ID : "+str(s_id)) # don't chage this code!
print(f"Decision: 'will VB exceed {PRE_ALARM} mm within {HORIZON} parts?'")
print(f"  correctly flagged (TP)      : {tp}")
print(f"  MISSED - ran a worn tool (FN): {fn}")
print(f"  false alarm (FP)            : {fp}")
print(f"  correctly quiet (TN)        : {tn}")
if tp + fn:
    print(f"\n  recall    {tp/(tp+fn):.3f}   (fraction of real cases caught)")
if tp + fp:
    print(f"  precision {tp/(tp+fp):.3f}   (fraction of alarms that were real)")

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. In Part F, the summary-statistics MLP scored almost exactly the same on tool-change windows as on ordinary ones. Explain why that near-equality is the expected result for an order-blind model, rather than a coincidence.**

*Your answer:*

**2. Describe, in terms of what the hidden state carries, why a shuffled window makes the LSTM's forecast so much worse specifically on tool-change windows.**

*Your answer:*

**3. We built training and test windows from two separate machine logs. Suppose a colleague instead pooled every window and split them 80/20 at random. Explain concretely why their reported error would be far too optimistic, and name the Week 6 concept involved.**

*Your answer:*

**4. Look at your decision table in Part G. Which error type is more expensive for a machining line — a missed tool change, or a false alarm? How would you shift the threshold (currently 90% of the limit) to reflect that, and what would you give up? (Recall Week 6.)**

*Your answer:*

**5. Change `WINDOW` from 60 to 30 at the top of §3 and re-run the notebook. The LSTM's advantage over the summary-statistics MLP largely disappears. Explain why a shorter window favours the order-blind model. (Think about what fraction of the information in 30 numbers survives being reduced to four statistics, versus 60 numbers.)**

*Your answer:*

**6. Suppose the signal that predicts a catastrophic failure actually begins 300 parts earlier. Explain why simply setting `WINDOW = 300` would be a poor fix for a plain recurrent network, and name the specific training problem it would run into.**

*Your answer:*

## 11. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The machine-log plot (§3), the shuffle comparison table (§7), the error-by-window-type chart (§8), and the forecast plots and decision table (§9).
- Written answers to the five reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| `make_windows` correctly implemented (no off-by-one, no leakage) | 20% |
| `WearLSTM.forward` correct; shape check passes | 20% |
| Shuffle experiment correctly constructed and its control understood | 25% |
| Per-window-type analysis and the maintenance decision in §9 interpreted | 20% |
| Reflection questions (including the WINDOW experiment in Q5) | 10% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 5% |

**Note :** Actual weights are subjected to Grader's discretion.

Each output cell that shows incorrect s_id printout will get -1 penalty per cell.



---
**Next week:** *Attention & Transformers* — why a fixed-size hidden state becomes the bottleneck on long sequences, and what replaced it.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>